In [2]:
import snapatac2 as snap
import numpy as np
import polars as pl

import os
snap.__version__

'2.8.0'

In [3]:
work_dir = '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/motor-pathway_multiome_cellbender_cluster-snrna-cr/'
pp_name = 'snapatac_preprocess'
data_dir = os.path.join(work_dir, pp_name)
data_sub_fname = os.path.join(data_dir,  "subset.h5ad")

script_name = "call_peaks"
out_dir = os.path.join(work_dir, script_name)
os.makedirs(out_dir, exist_ok=True)
out_fname = os.path.join(out_dir, "data.h5ad")

In [4]:
data = snap.read(filename = data_sub_fname)

thread '<unnamed>' panicked at /root/.cargo/registry/src/index.crates.io-6f17d22bba15001f/pyanndata-0.4.1/src/anndata.rs:36:60:
called `Result::unwrap()` on an `Err` value: H5Fopen(): unable to open file: unable to lock file, errno = 11, error message = 'Resource temporarily unavailable'
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace


PanicException: called `Result::unwrap()` on an `Err` value: H5Fopen(): unable to open file: unable to lock file, errno = 11, error message = 'Resource temporarily unavailable'

In [ ]:
snap.tl.macs3(data, groupby='cluster', tempdir=out_dir, n_jobs=31)

2026-03-26 22:14:03 - INFO - Exporting fragments...


In [ ]:
data.uns['macs3']

NameError: name 'data' is not defined

In [ ]:
chrom_sizes = dict(zip(
    data.uns['reference_sequences']['reference_seq_name'].to_list(),
    data.uns['reference_sequences']['reference_seq_length'].to_list()
))

data.uns['macs3'] = {
    key: df.with_columns(
        pl.struct(['chrom', 'end']).map_elements(
            lambda r: min(r['end'], chrom_sizes[r['chrom']]),
            return_dtype=pl.Int64
        ).alias('end')
    ).unique(subset=['chrom', 'start', 'end'])
    for key, df in data.uns['macs3'].items()
}

In [ ]:
import matplotlib.pyplot as plt

clusters = list(data.uns['macs3'].keys())
widths = [(df['end'] - df['start']).to_numpy() for df in data.uns['macs3'].values()]

fig, ax = plt.subplots(figsize=(max(6, len(clusters) * 0.6), 4))
ax.boxplot(widths, tick_labels=clusters, showfliers=False)
ax.set_xlabel('Cluster')
ax.set_ylabel('Peak width (bp)')
ax.set_title('Peak width distribution by cluster')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
for key, df in data.uns['macs3'].items():
    bed_path = os.path.join(out_dir, f"{key}.bed")
    df.select(['chrom', 'start', 'end']).sort(['chrom', 'start']).write_csv(bed_path, separator='\t', include_header=False)

In [ ]:
data.write(out_fname)